<h1 style='font-weight:bold; color:#003366; text-align:center;'>Tema 13: Estructura de proyectos y entornos virtuales</h1>

<h3 style='font-weight:bold; color:#003366; text-align:center;'>Programación Analítica II</h3>
<h4 style='font-weight:bold; color:#003366; text-align:center;'>Docente Camila Palomeque</h4>
<h4 style='font-weight:bold; color:#003366; text-align:center;'>S2 . 2026</h4>

<h2 style='font-weight:bold; color:#003366;'>2. La estructura del proyecto final del curso</h2>

```
analisis-finco-APELLIDO/
├── README.md              # La explicación del proyecto para alguien que llega de cero
├── requirements.txt        # Las librerías que el análisis necesita, con su versión
├── .gitignore              # qué NO va al repositorio
├── datos/
│   └── (los CSV, ignorados por git)
├── notebooks/
│   └── proyecto_final.ipynb      # El notebook con los análisis realizados
└── src/ (opcional)
    ├── carga.py            # script con funciones para leer y limpiar
    └── analisis.py         # script con funciones para realizar cálculos repetitivos
```

**`datos/` es intocable.** Los archivos que te mandaron se guardan como llegaron y no se editan. Toda la limpieza se hace **con código**.


In [2]:
import os
from pathlib import Path

# Anclamos la carpeta del notebook para poder volver más adelante
CARPETA_NOTEBOOK = Path.cwd()
CARPETA_SALIDAS = CARPETA_NOTEBOOK / "salidas"
PROYECTO_DEMO = CARPETA_SALIDAS / "demo_proyecto"

for subcarpeta in ["datos", "notebooks", "src"]:
    (PROYECTO_DEMO / subcarpeta).mkdir(parents=True, exist_ok=True)

# Los archivos de la raíz, por ahora vacíos: los vamos completando durante la clase
for archivo in ["README.md", "requirements.txt", ".gitignore"]:
    (PROYECTO_DEMO / archivo).touch()

print(f"{PROYECTO_DEMO.name}/")
for ruta in sorted(PROYECTO_DEMO.rglob("*")):
    nivel = len(ruta.relative_to(PROYECTO_DEMO).parts) - 1
    print("    " * nivel + "|-- " + ruta.name + ("/" if ruta.is_dir() else ""))

print("\nEsto mismo, en la terminal, se hace con: mkdir datos notebooks src")

demo_proyecto/
|-- .gitignore
|-- datos/
|-- notebooks/
|-- README.md
|-- requirements.txt
|-- src/

Esto mismo, en la terminal, se hace con: mkdir datos notebooks src


La celda de arriba creó la estructura dentro de `salidas/demo_proyecto/`, para no ensuciar la
carpeta del curso. **En tu proyecto real, esas tres carpetas van en la raíz del repositorio que
subiste la clase pasada.**

Podés crearlas con el explorador de archivos, con el editor, o con `mkdir` en la terminal. Da
exactamente lo mismo: son carpetas.

<h2 style='font-weight:bold; color:#003366;'>3. Módulos: usar tu propio código desde el notebook</h2>

Un **módulo** es un archivo `.py`. Eso es todo: no hay que declarar nada especial ni instalar nada.
Si el archivo existe y Python sabe dónde buscarlo, se puede importar.

Vamos a hacerlo de verdad. Primero escribimos un módulo chico con dos funciones que el proyecto
final necesita sí o sí, y después lo usamos desde el notebook.

La primera función resuelve un problema real del caso FinCo: `monto_otorgado` viene **en la moneda
del contrato**, así que hay pesos y dólares en la misma columna y sumarlos directamente da un
número sin sentido. La segunda calcula la mora, que como sabemos no viene en los datos.

In [3]:
%%writefile salidas/demo_proyecto/datos/cartera_demo.csv
id_credito,codigo_sucursal,moneda,monto_otorgado,dias_atraso
C-0001,S01,UYU,120000,0
C-0002,S01,USD,3000,0
C-0003,S02,UYU,85000,12
C-0004,S02,UYU,240000,105
C-0005,S03,USD,1500,45
C-0006,S03,UYU,60000,0
C-0007,S01,UYU,310000,210
C-0008,S04,USD,5200,0
C-0009,S04,UYU,95000,63
C-0010,S02,UYU,150000,0
C-0011,S03,UYU,45000,98
C-0012,S04,USD,2800,7

Writing salidas/demo_proyecto/datos/cartera_demo.csv


In [4]:
%%writefile salidas/demo_proyecto/src/carga.py
"""Lectura de la cartera de créditos de FinCo."""

import csv

# Pesos uruguayos por dólar al 30/06/2026, fecha de corte de los datos
TIPO_DE_CAMBIO_DE_CORTE = 41.5


def cargar_cartera(ruta_archivo):
    """Lee el CSV de la cartera y agrega el monto expresado en pesos.

    El monto viene en la moneda del contrato, así que sin convertir no se puede sumar.
    """
    creditos = []
    with open(ruta_archivo, encoding="utf-8") as archivo:
        for fila in csv.DictReader(archivo):
            fila["monto_otorgado"] = float(fila["monto_otorgado"])
            fila["dias_atraso"] = int(fila["dias_atraso"])
            if fila["moneda"] == "USD":
                fila["monto_en_pesos"] = fila["monto_otorgado"] * TIPO_DE_CAMBIO_DE_CORTE
            else:
                fila["monto_en_pesos"] = fila["monto_otorgado"]
            creditos.append(fila)
    return creditos

Writing salidas/demo_proyecto/src/carga.py


Ahora hay que importarlo, y acá aparece la única complicación de todo el bloque.

Python busca los módulos **a partir de la carpeta desde la que estás trabajando**. Si el notebook
está parado en `notebooks/`, desde ahí no ve la carpeta `src/`, que está un nivel arriba. Ese es el
famoso `No module named 'src'`.

La solución es de una línea y es la misma que van a usar en su proyecto: **pararse en la raíz del
proyecto**. Se hace una sola vez, en la primera celda del notebook.

```python
import os
os.chdir("..")     # el notebook está en notebooks/, subimos a la raíz del proyecto
```

Acá abajo hacemos exactamente eso, pero apuntando a la carpeta de la demo. De esta celda en
adelante, este notebook trabaja **como si fuera el notebook de tu proyecto**.

In [5]:
os.chdir(PROYECTO_DEMO)
print("Trabajando desde:", Path.cwd())
print("¿Ve la carpeta src?", (Path.cwd() / "src").exists())

Trabajando desde: c:\Users\Camila Palomeque\OneDrive - Quanam\UM_Clases\Prog_Analitica_II\Notebooks\salidas\demo_proyecto
¿Ve la carpeta src? True


In [6]:
# Ahora sí: importamos nuestro propio código igual que importaríamos pandas
from src.carga import cargar_cartera

cartera = cargar_cartera("datos/cartera_demo.csv")

print(f"Créditos cargados: {len(cartera)}")
print(f"Saldo total de la cartera: $ {sum(c['monto_en_pesos'] for c in cartera):,.0f}")

Créditos cargados: 12
Saldo total de la cartera: $ 1,623,750


**Importante**: Python **importa cada módulo una sola vez por sesión**: la primera vez lee el archivo
y se lo guarda en memoria, y a partir de ahí un `import` del mismo módulo no vuelve a leer el
disco, simplemente te devuelve la copia que ya tiene. 

Por lo que si modifican un script de python y quieren volver a importarlo tienen que resetear el Kernel con Restart

<h2 style='font-weight:bold; color:#003366;'>5. Separar parámetros de configuración del código</h2>

**Todo valor que alguien podría querer cambiar va arriba, con nombre en MAYÚSCULAS (idealmente) y en un solo lugar.**

En el proyecto final eso incluye, como mínimo:

```python
# Configuración del análisis
CARPETA_DATOS = Path("datos")
FECHA_DE_CORTE = "2026-06-30"          # la que dice el README del caso
DIAS_PARA_CONSIDERAR_MORA = 90         # convención del banco central
TIPO_DE_CAMBIO_DE_CORTE = 41.5         # pesos por dólar al 30/06/2026
CUOTA_INGRESO_MAXIMA = 0.35            # umbral de carga financiera que usamos
```

Dos ventajas:

1. **Documenta los supuestos.** Ese bloque es la lista de todas las decisiones que tomaste. Cuando
   el corrector pregunte "¿por qué 90 días?", la respuesta está ahí con su comentario al lado.
2. **Hace visible el análisis de sensibilidad.** Cambiar un umbral y volver a correr pasa a ser
   trivial, y entonces sí se puede contestar "¿cuánto cambia la conclusión si uso 30 días?".

<h2 style='font-weight:bold; color:#003366;'>6. El README.md</h2>

El `README.md` es lo primero que ve cualquiera que abra tu repositorio, incluido el que corrige.

Tiene que contestar cinco preguntas, y ninguna necesita más de un párrafo:

| Sección | La pregunta que contesta |
|---|---|
| De qué se trata | ¿Qué problema de negocio se está resolviendo, y para quién? |
| Los datos | ¿De dónde vienen, qué período cubren, dónde hay que ponerlos? |
| Cómo se instala | ¿Qué comandos corro para poder ejecutarlo? |
| Cómo se corre | ¿Qué archivo abro y en qué orden? |
| Qué se concluyó | ¿Cuáles son los tres o cuatro hallazgos? |

`.md` es **Markdown**, el mismo formato de las celdas de texto de este notebook: `#` para títulos,
`**negrita**`, `-` para listas. GitHub lo muestra formateado en la portada del repositorio.

<h2 style='font-weight:bold; color:#003366;'>Entornos virtuales</h2>

In [7]:
os.chdir(CARPETA_NOTEBOOK)
print("De vuelta en:", Path.cwd())

De vuelta en: c:\Users\Camila Palomeque\OneDrive - Quanam\UM_Clases\Prog_Analitica_II\Notebooks


<h2 style='font-weight:bold; color:#003366;'>7. El problema que resuelven los entornos virtuales</h2>

Al trabajar en Python y le vas agregando librerías: pandas para un trabajo, plotly para otro, después una
librería de gráficos que necesita una versión vieja de numpy. Todo se acumula en **la misma
instalación**, la global, la que usa toda la computadora y esto a la larga genera errores o incompatibilidades.

Un **entorno virtual** es la solución, y es más simple de lo que parece: es **una carpeta** dentro
de tu proyecto que contiene su propia copia de Python y sus propias librerías. Cada proyecto tiene
la suya. "Activar" el entorno significa decirle a la terminal "cuando escriba `python` o `pip`,
usá los de esta carpeta". 

Dos consecuencias:

1. Si borrás la carpeta, el entorno desaparece sin dejar rastro. No rompe nada.
2. Por eso mismo **la carpeta del entorno nunca se sube al repositorio**: se reconstruye con un
   comando desde `requirements.txt`.

<h2 style='font-weight:bold; color:#003366;'>8. venv paso a paso</h2>

`venv` viene incluido con Python, así que no hay que instalar nada. Son tres comandos: crear,
activar, desactivar.

**Crear** se hace una sola vez por proyecto, con la terminal parada en la carpeta del proyecto.
**Activar** se hace cada vez que abrís una terminal nueva para trabajar.

| Sistema y terminal | Crear (una vez) | Activar (siempre) |
|---|---|---|
| **Windows, PowerShell** | `python -m venv .venv` | `.venv\Scripts\Activate.ps1` |
| **Windows, cmd** | `python -m venv .venv` | `.venv\Scripts\activate.bat` |
| **Windows, Git Bash** | `python -m venv .venv` | `source .venv/Scripts/activate` |
| **macOS y Linux** | `python3 -m venv .venv` | `source .venv/bin/activate` |

**Desactivar**, en todos los casos: `deactivate`.

Sobre el nombre `.venv`: es la convención más extendida. El punto adelante la hace oculta y todos
los editores la reconocen automáticamente. Podés llamarla `venv`, `env` o un nombre más amigable como `progII`, pero cualquiera sea el
nombre, **tiene que estar en el `.gitignore`**.

### Cómo saber si funcionó

Cuando el entorno está activado, el nombre aparece **entre paréntesis al principio de la línea**:

```
(.venv) C:\Users\cami\proyectos\analisis-prog-analitica>
(.venv) cami@MacBook analisis-prog-analitica %
```

Ese `(.venv)` es la única señal, y es la que hay que mirar antes de correr cualquier `pip install`.
Si no está, `pip` va a instalar en el Python del sistema y el entorno va a quedar vacío mientras vos
crees que instalaste todo. 

<h2 style='font-weight:bold; color:#003366;'>9. pip, los cuatro comandos que se usan</h2>

`pip` es el programa que instala librerías. Descarga de **PyPI**, el repositorio oficial de
paquetes de Python. Con el entorno activado, todo lo que instale va a la carpeta del entorno y a
ningún otro lado.

Los que se usan de verdad son cuatro, más uno:

| Comando | Qué hace |
|---|---|
| `pip install pandas` | Instala la última versión disponible |
| `pip install pandas==2.2.3` | Instala **esa** versión exacta |
| `pip list` | Lista todo lo que hay instalado **en este entorno** |
| `pip show pandas` | Detalles de un paquete: versión, dónde está instalado y de qué depende |
| `pip uninstall pandas` | Lo desinstala |
| `pip install -r requirements.txt` | Instala todo lo que dice el archivo, de una vez |

El último es el que hace reproducible un proyecto y es el que va en el README.

Dos cosas que conviene saber para no asustarse:

**Cuando instalás una librería aparecen varias.** `%pip install pandas` instala también `numpy`,
`python-dateutil`, `pytz` y `tzdata`, que son las que pandas necesita para funcionar. Se llaman
**dependencias transitivas** y pip las resuelve solo. Eso explica por qué `pip list` muestra
treinta paquetes cuando vos instalaste cuatro.

<h2 style='font-weight:bold; color:#003366;'>10. requirements.txt</h2>

`requirements.txt` es un archivo de texto, en la raíz del proyecto, con una librería por línea.
Eso es todo. Su función es que otra persona pueda poner su máquina en el mismo estado que la tuya
con un comando.

Se puede generar automáticamente con `pip freeze > requirements.txt`

O podés hacerlo a mano para solo poner los paquetes que instalaste y no todas sus dependencias que igual ya se instalan. Para eos corré `pip list` con el entorno activado y copiá los números que
tenés funcionando. 

<h2 style='font-weight:bold; color:#003366;'>11. El kernel de Jupyter y el entorno</h2>

Y acá llega el problema más confuso de toda la clase. Se da así:

> Activás el entorno en la terminal, ves el `(.venv)`, corrés `pip install pandas`, dice
> *Successfully installed*. Abrís el notebook, escribís `import pandas` y te dice
> **`ModuleNotFoundError: No module named 'pandas'`**.

Esto pasa porque **el notebook no usa la terminal**. Un
notebook está conectado a un **kernel**, que es un proceso de Python aparte, elegido cuando se abrió
el notebook. Activar un entorno en la terminal no cambia el kernel de un notebook que ya está
abierto, ni de uno que se abre después: son dos mundos separados.

La solución es registrar el entorno como un kernel de Jupyter, y después elegirlo. Con el entorno
**activado**:

```bash
pip install ipykernel
python -m ipykernel install --user --name ProgII --display-name "Python (ProgII)"
```

Después, en el notebook:

- **En VS Code:** arriba a la derecha dice *Select Kernel*. Elegí *Python (ProgII)*.
- **En Jupyter Lab o Notebook:** menú *Kernel → Change Kernel* y elegí *Python (ProgII)*.

Y la forma de verificar que quedó bien no es la intuición, es una celda: `sys.executable` te dice
qué Python está usando el notebook en este instante. Si la ruta pasa por tu `.venv`, está bien. Si
no, el kernel es otro.

**En Colab esto no aplica y conviene decirlo claro.** Colab no usa entornos virtuales, porque cada
sesión de Colab **ya es** un entorno aislado y temporal que se destruye al cerrarla. Armar un
`venv` adentro se puede, pero no aporta nada y complica todo. El patrón correcto en Colab es una
celda de instalación al principio del notebook:

```python
!pip install --quiet -r requirements.txt
```

Dos advertencias de Colab que valen para el proyecto:

- Colab ya trae pandas, numpy, matplotlib, seaborn y plotly. Solo instalá lo que falte.
- Si instalás una versión **distinta** de algo que Colab ya tenía cargado, hay que reiniciar el
  entorno de ejecución (*Entorno de ejecución → Reiniciar sesión*) para que el cambio tome efecto.

<h2 style='font-weight:bold; color:#003366;'>Tutorial guiado: armá el entorno de tu proyecto</h2>

El trabajo de la clase, sobre **tu** proyecto. Se hace en paralelo con la docente.

### Parte A: la estructura y el código propio

- [ ] **1.** En la carpeta de tu proyecto (el repositorio de la clase pasada), creá las tres
  carpetas: `datos/`, `notebooks/` y `src/`.
- [ ] **2.** Mové los CSV del caso a `datos/` y el notebook del análisis a `notebooks/`.
- [ ] **3.** Confirmá que `datos/*.csv` está en el `.gitignore`. Si no, agregalo.
- [ ] **4.** Creá `src/carga.py` y mudá ahí la función que lee los archivos. Si todavía no la
  tenés como función, este es el momento.
- [ ] **5.** En la primera celda del notebook, poné las tres cosas que van siempre:
  `import os; os.chdir("..")`, `%load_ext autoreload` con `%autoreload 2`, y los `import`.
- [ ] **6.** Escribí el `README.md` con las cinco secciones. Los hallazgos pueden quedar
  incompletos por ahora.

### Parte B: el entorno virtual

- [ ] **7.** Abrí una terminal **dentro** de la carpeta del proyecto.
- [ ] **8.** Creá el entorno: `python -m venv .venv` (en macOS, `python3 -m venv .venv`).
- [ ] **9.** Activalo con el comando de tu sistema (tabla de la sección 8). **Confirmá que
  aparece `(.venv)` al principio de la linea.** Si no aparece, no sigas: no está activado.
- [ ] **10.** Instalá lo que usa tu análisis: `pip install pandas matplotlib seaborn plotly
  jupyter ipykernel`.
- [ ] **11.** Corré `pip list` y escribí el `requirements.txt`, copiando las versiones
  que te muestra.
- [ ] **12.** Confirmá que `.venv/` está en el `.gitignore`. Corré `git status`: la carpeta del
  entorno **no** tiene que aparecer.
- [ ] **13.** Registrá el kernel:
  `python -m ipykernel install --user --name progIIproyecto --display-name "Python (progIIproyecto)"`.
- [ ] **14.** En el notebook, elegí ese kernel y corré una celda con
  `import sys; print(sys.executable)`. Tiene que aparecer tu `.venv` en la ruta.

### Parte C: la verificación

- [ ] **15.** *Kernel → Restart Kernel and Run All Cells*. El notebook tiene que correr entero,
  sin errores y sin celdas salteadas.

<h2 style='font-weight:bold; color:#003366;'>Diagnóstico de errores frecuentes</h2>

| Problema | Causa | Solución |
|---|---|---|
| `No se puede cargar el archivo Activate.ps1 porque la ejecución de scripts está deshabilitada` | Política de seguridad de PowerShell | `Set-ExecutionPolicy -ExecutionPolicy RemoteSigned -Scope CurrentUser`, una sola vez, y volvé a activar |
| No aparece `(.venv)` en el prompt | La activación falló, o corriste el comando de otro sistema operativo | Revisá la tabla de la sección 8. En PowerShell es `.venv\Scripts\Activate.ps1`, en macOS es `source .venv/bin/activate` |
| `source: command not found` o `source no se reconoce` en Windows | `source` es de bash, en PowerShell y cmd no existe | Usá `.venv\Scripts\Activate.ps1` (PowerShell) o `.venv\Scripts\activate.bat` (cmd) |
| `python` no se reconoce como comando (Windows) | Python no quedó en el PATH al instalarse | Probá `py -3 -m venv .venv`. Si tampoco anda, reinstalá Python marcando **Add Python to PATH** |
| Escribo `python` y se abre la Microsoft Store (Windows) | Es el alias que Windows pone cuando no hay Python instalado | Usá `py` en lugar de `python`, o desactivá el alias en *Configuración → Aplicaciones → Alias de ejecución de aplicaciones* |
| `python: command not found` (macOS) | En macOS el comando es `python3`, `python` a secas no existe | `python3 -m venv .venv` y `pip3` si hace falta |
| Moví o renombré la carpeta del proyecto y el entorno dejó de funcionar | Un `venv` guarda rutas absolutas adentro: si la carpeta cambia de lugar, se rompe | Borrá `.venv` y recreálo. Es un comando y treinta segundos |
| El entorno pesa 700 MB | Normal, sobre todo con matplotlib y jupyter | No es un problema. Lo único que importa es que esté en el `.gitignore` |
| Borré `.venv` sin querer | No pasa nada grave, para eso están los entornos | `python -m venv .venv`, activar, `pip install -r requirements.txt` |
| `.venv` se subió a Git | No estaba en el `.gitignore` cuando hiciste el primer commit | `git rm -r --cached .venv`, agregar `.venv/` al `.gitignore`, commit y push |
| `ModuleNotFoundError: No module named 'pandas'` y estoy seguro de haberlo instalado | Instalaste en un entorno y estás ejecutando en otro | Corré `import sys; print(sys.executable)`. Si la ruta no pasa por tu `.venv`, ahí está el problema |
| El entorno está activado en la terminal pero el notebook usa otro Python | El notebook no usa la terminal: usa su **kernel** | Registrá el kernel con `ipykernel` (sección 11) y elegilo en el notebook |
| `pip install` dice *Successfully installed* pero el import falla | Instalaste en el Python global porque el entorno no estaba activado | Mirá el prompt: si no dice `(.venv)`, activá primero. Y usá `python -m pip install` |
| `pip` instala en el Python global aunque activé el entorno | Hay varias versiones de Python en la máquina y `pip` es el de otra | Siempre `python -m pip install`, nunca `pip install` a secas cuando hay dudas |
| `ModuleNotFoundError: No module named 'src'` | El notebook está corriendo desde `notebooks/` y desde ahí no ve la carpeta `src/`, que está un nivel arriba | `import os; os.chdir("..")` en la primera celda del notebook. O abrí el editor en la carpeta raíz del proyecto |
| Edité `src/carga.py` y el notebook sigue usando la versión vieja | Python importa cada módulo una sola vez por sesión | `%load_ext autoreload` y `%autoreload 2` en la primera celda. Si ya es tarde, reiniciá el kernel |
| Puse `autoreload` y tampoco toma el cambio | `autoreload` revisa los archivos **al empezar** cada celda: si editás y usás en la misma celda, no llega | Poné la edición y el uso en celdas distintas, o reiniciá el kernel |
| `ERROR: Could not find a version that satisfies the requirement` | El nombre está mal escrito, la versión no existe, o esa versión no es compatible con tu Python | Buscá el nombre exacto en pypi.org. Probá sin fijar versión para ver qué instala, y después fijala |
| Después de activar el entorno, `jupyter` no existe | Jupyter estaba instalado en el Python global, no en el entorno | `pip install jupyter ipykernel` dentro del entorno. O usá VS Code, que solo necesita `ipykernel` |
| El notebook corre en mi máquina y falla en la de mi compañero | Una librería que no está en `requirements.txt`, o una ruta absoluta | Hacé la prueba de reproducibilidad de la sección 12 antes de entregar |
| `FileNotFoundError: datos/cartera_creditos.csv` en la máquina del otro | Los datos no están en el repositorio (y está bien que no estén), pero el README no dice dónde ponerlos | Documentalo en el README: de dónde se bajan y en qué carpeta van |
| `FileNotFoundError` con una ruta que empieza en `C:\Users\...` | Ruta absoluta: esa carpeta solo existe en tu computadora | Rutas relativas siempre: `datos/cartera_creditos.csv` |
| El notebook da números distintos que ayer y no cambié nada | Ejecutaste las celdas en otro orden, y quedó estado viejo en memoria | *Restart Kernel and Run All Cells*. Si los números cambian, el notebook tenía un problema de orden |